In [1]:
# ============================================================
# HTTP PAYLOAD SALDIRI SINIFLANDIRICISI (LAYER 2) - v2
# WEB_APPLICATION_PAYLOADS.jsonl tabanli - DOGRUDAN ETIKETLI
# (supervised) SQLi / XSS / CSRF / SSRF / Command Injection
#
# BU SCRIPT NEDEN v1'DEN (CSIC2010 tabanli) FARKLI?
# ------------------------------------------------------------
# Onceki versiyon (payload_classifier_csic2010.py), CSIC2010
# datasetinin yalnizca "normal/anomalous" etiketi tasidigi icin
# SQLi/XSS ayrimini REGEX TABANLI ZAYIF ETIKETLEME (weak
# supervision) ile kendimiz uretmek zorunda kalmistik.
#
# WEB_APPLICATION_PAYLOADS.jsonl ise ZATEN DOGRU ETIKETLENMIS:
# her kaydin "id" alani (sqli-xxx, xss-xxx, csrf-xxx, ssrf-xxx,
# cmdinj-xxx) payload'in hangi saldiri turune ait oldugunu
# ACIKCA belirtiyor. Bu sayede regex ile zayif etiketleme
# adimina GEREK KALMADI - bu GERCEK (ground-truth) etiketli
# bir supervised ogrenme problemi.
#
# ONEMLI - DOSYA FORMATI HAKKINDA (".jsonl" YANILTICI OLABILIR):
# ------------------------------------------------------------
# Dosya ".jsonl" uzantili olsa da GERCEK JSONL (satir basina
# bagimsiz JSON objesi) formatinda OLMAYABILIR; GitHub'daki
# surumde "[" ile baslayan, objelerin "},  {" seklinde ayrildigi
# TEK BUYUK bir JSON array'i olabilir (pretty-print edilmis).
# Bu yuzden BRACE-COUNTING tabanli bir ayiklayici kullaniyoruz
# (asagidaki extract_json_objects()): ham metni karakter
# karakter tarayip süslü parantez ({ ve }) derinligini SAYARAK
# her UST-SEVIYE JSON objesini bagimsiz bir blok olarak ayiklar.
# Bu yontem:
#   1) Gercek JSONL ile pretty-print JSON array arasinda fark
#      gozetmez.
#   2) Objeler arasinda virgul olsun ya da olmasin calisir.
#   3) Tek tek bozuk objeler (obje icinde eksik virgul, gecersiz
#      kacis/escape karakteri gibi) SADECE o objeyi etkiler;
#      diger saglam objeler kaybolmaz.
#
# ONEMLI - HTML ARTEFAKT TEMIZLIGI:
# ------------------------------------------------------------
# Bazen dosyanin bir kopyasi (ozellikle bir web sayfasindan
# veya zengin metin destekleyen bir arayuzden kopyala-yapistir
# yoluyla elde edilmisse) icinde otomatik olarak "linkified"
# edilmis URL'ler bulunabilir. strip_html_artifacts() fonksiyonu
# bu tur "<a ...>...</a>" etiketlerini tespit edip yalnizca IC
# METNI birakarak temizler.
#
# ONEMLI - EKSIK/BOS PAYLOAD ALANLARI:
# ------------------------------------------------------------
# load_payload_json(), basariyla parse edilen kayitlar arasinda
# "payload" alani eksik, None veya bos olan satirlari ACIKCA
# tespit edip (log'layarak) egitim disinda birakir. Bu, TF-IDF
# vectorizer'in "np.nan is an invalid document" hatasi vermesini
# kaynaginda onler.
#
# ONEMLI - OZEL CLASS_WEIGHT (BU SURUMDE YENI):
# ------------------------------------------------------------
# Ilk egitimlerde "Command Injection" sinifinin, kisa/genel
# kabuk meta-karakterleri (;, &, |, $() gibi) tasidigi icin
# SSRF, XSS ve CSRF'den bazi kayitlari "cektigi" (yanlis pozitif
# aldigi) gozlemlendi (precision ~0.91-0.9126). Bunu azaltmak
# icin CUSTOM_CLASS_WEIGHTS_BY_NAME sozlugu ile Command
# Injection'in agirligini dusuruyor, SSRF/CSRF/XSS'e hafif
# ekstra agirlik veriyoruz.
#
# ONEMLI TEKNIK NOT: sklearn'in class_weight parametresi bir
# sozluk olarak verildiginde, ANAHTARLARIN (key) kategori ISMI
# DEGIL, LabelEncoder'in urettigi TAM SAYI (int) indeksi olmasi
# GEREKIR (ornegin {'Command Injection': 0.7} calismaz, ama
# {1: 0.7} calisir - eger 'Command Injection' LabelEncoder'da
# indeks 1'e karsilik geliyorsa). Bu yuzden asagida once ISIM
# bazli bir sozluk taniyoruz (okunabilir olmasi icin), sonra
# bunu label_encoder.classes_ sirasina gore INT bazli bir
# sozluge CEVIRIYORUZ (bkz. train_payload_classifier_v2 icinde
# class_weight_dict olusturulan blok).
# ============================================================

import os
import re
import gc
import json
import warnings
from pathlib import Path

import joblib
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.ensemble import ExtraTreesClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    accuracy_score,
    balanced_accuracy_score,
    f1_score
)


warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", 200)
pd.set_option("display.max_colwidth", 160)
pd.set_option("display.width", 220)


def log(*args):
    """Mesajlari aninda terminale yazdirir."""
    print(*args, flush=True)


# ============================================================
# 1. GENEL AYARLAR
# ============================================================

RANDOM_STATE = 42

N_SPLITS_MAX = 5
N_SPLITS_MIN = 2

CV_N_ESTIMATORS = 200
FINAL_N_ESTIMATORS = 500

N_JOBS = 4

TFIDF_NGRAM_RANGE = (2, 5)
TFIDF_MAX_FEATURES = 20000

LOW_CONFIDENCE_THRESHOLD = 0.40

# id onekinden okunabilir kategori ismine esleme.
ID_PREFIX_TO_CATEGORY = {
    "sqli": "SQL Injection",
    "xss": "XSS",
    "csrf": "CSRF",
    "ssrf": "SSRF",
    "cmdinj": "Command Injection"
}

# ------------------------------------------------------------
# OZEL CLASS_WEIGHT AYARLARI (ISIM BAZLI - OKUNABILIR HALI)
# ------------------------------------------------------------
# Command Injection'in agirligini dusuruyoruz (once 0.9126
# precision aliyordu, digerlerinden yanlis pozitif "cekiyordu").
# SSRF/CSRF/XSS'e hafif ekstra agirlik veriyoruz cunku bu
# siniflar Command Injection'a en cok kayit kaptiran siniflardi
# (confusion matrix'e gore SSRF: 5 kayit, XSS: 3 kayit,
# CSRF: 2 kayit Command Injection'a yanlis atanmisti).
#
# Bu degerler baslangic noktasidir; modeli tekrar calistirip
# confusion matrix'e baktiktan sonra ince ayar yapabilirsiniz
# (ornegin sizinti devam ediyorsa Command Injection'i 0.5'e
# dusurebilirsiniz).
CUSTOM_CLASS_WEIGHTS_BY_NAME = {
    "CSRF": 1.2,
    "Command Injection": 0.7,
    "SQL Injection": 1.0,
    "SSRF": 1.2,
    "XSS": 1.1,
}

# Layer 1'in WebAttackCandidate kapali kumesi (SQLi/XSS/
# BruteForce) ile uyumluluk icin, bu kumenin DISINDA kalan
# tahminler (CSRF/SSRF/CmdInj) VE dusuk guvenli tahminler,
# eleme yontemiyle bu etikete eslenir.
FALLBACK_FINAL_LABEL = "Web Brute Force"

DECISION_SOURCE_ML = "ml_model"
DECISION_SOURCE_FALLBACK = "elimination_fallback"


# ============================================================
# 2. DOSYA VE KLASOR AYARLARI
# ============================================================

DATA_DIR = "data"

PAYLOAD_JSONL_FILE = os.path.join(DATA_DIR, "WEB_APPLICATION_PAYLOADS.jsonl")
PAYLOAD_JSON_FILE_FALLBACK = os.path.join(DATA_DIR, "WEB_APPLICATION_PAYLOADS.json")

MODEL_DIR = "models"
OUTPUT_DIR = "outputs"

os.makedirs(MODEL_DIR, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)

PAYLOAD_MODEL_PATH = os.path.join(MODEL_DIR, "payload_attack_classifier_v2.joblib")
PAYLOAD_REPORT_PATH = os.path.join(OUTPUT_DIR, "payload_v2_classification_report_oof.csv")
PAYLOAD_FOLD_METRICS_PATH = os.path.join(OUTPUT_DIR, "payload_v2_fold_metrics.csv")
PAYLOAD_CONFUSION_MATRIX_PATH = os.path.join(OUTPUT_DIR, "payload_v2_confusion_matrix_oof.png")
PAYLOAD_CATEGORY_SUMMARY_PATH = os.path.join(OUTPUT_DIR, "payload_v2_category_summary.csv")
PAYLOAD_TRAINING_DATA_PATH = os.path.join(OUTPUT_DIR, "payload_v2_training_dataset.csv")
PAYLOAD_SANITY_CHECK_PATH = os.path.join(OUTPUT_DIR, "payload_v2_sanity_check.csv")
PAYLOAD_PARSE_ERRORS_PATH = os.path.join(OUTPUT_DIR, "payload_v2_parse_errors_debug.txt")
PAYLOAD_DROPPED_ROWS_PATH = os.path.join(OUTPUT_DIR, "payload_v2_dropped_rows_debug.csv")


# ============================================================
# 3. HTML ARTEFAKT TEMIZLIGI (ONCE-ISLEME)
# ============================================================

_ANCHOR_TAG_PATTERN = re.compile(
    r'<a\s+[^>]*>(.*?)</a>',
    re.IGNORECASE | re.DOTALL
)


def strip_html_artifacts(raw_text):
    """
    Ham metin icinde, bir web sayfasindan/zengin metin arayuzunden
    kopyala-yapistir sirasinda yanlislikla eklenmis olabilecek
    "<a href=\"...\" ...>IC_METIN</a>" ankor etiketlerini tespit
    edip yalnizca IC_METIN'i birakarak temizler.
    """

    if "<a " not in raw_text and "<a\n" not in raw_text:
        return raw_text, 0

    cleaned_text, count = _ANCHOR_TAG_PATTERN.subn(r"\1", raw_text)

    return cleaned_text, count


# ============================================================
# 4. BRACE-COUNTING TABANLI JSON/JSONL AYIKLAYICI (SAGLAM)
# ============================================================

def extract_json_objects(raw_text):
    """
    Ham metin icindeki tum ust-seviye JSON objelerini ({ ... })
    süslü parantez SAYARAK (brace-counting) tek tek ayiklar ve
    her birini BAGIMSIZ olarak parse etmeyi dener.

    Dondurur: (basarili_kayitlar, hatali_kayitlar)
    """

    objects = []
    depth = 0
    start_index = None
    in_string = False
    escape_next = False

    for index, char in enumerate(raw_text):

        if escape_next:
            escape_next = False
            continue

        if char == "\\":
            escape_next = True
            continue

        if char == '"':
            in_string = not in_string
            continue

        if in_string:
            continue

        if char == "{":
            if depth == 0:
                start_index = index
            depth += 1

        elif char == "}":
            depth -= 1
            if depth == 0 and start_index is not None:
                objects.append(raw_text[start_index:index + 1])
                start_index = None

    parsed_objects = []
    parse_errors = []

    for obj_index, obj_text in enumerate(objects):
        try:
            parsed_objects.append(json.loads(obj_text))
        except json.JSONDecodeError as e:
            preview = obj_text[:200].replace("\n", " ")
            parse_errors.append((obj_index, str(e), preview))

    return parsed_objects, parse_errors


def resolve_payload_file_path():
    """Once .jsonl, bulunamazsa .json uzantili dosyayi arar."""

    if os.path.exists(PAYLOAD_JSONL_FILE):
        return PAYLOAD_JSONL_FILE

    if os.path.exists(PAYLOAD_JSON_FILE_FALLBACK):
        return PAYLOAD_JSON_FILE_FALLBACK

    raise FileNotFoundError(
        "Payload dosyasi bulunamadi. Su yollardan birine "
        f"kopyalayin:\n  {PAYLOAD_JSONL_FILE}\n  {PAYLOAD_JSON_FILE_FALLBACK}"
    )


def load_payload_json(file_path=None):
    """
    WEB_APPLICATION_PAYLOADS dosyasini yukler ve bir pandas
    DataFrame olarak dondurur.
    """

    if file_path is None:
        file_path = resolve_payload_file_path()

    log("\n" + "=" * 80)
    log("Payload JSON/JSONL dosyasi okunuyor:", file_path)

    if not os.path.exists(file_path):
        raise FileNotFoundError(f"Payload dosyasi bulunamadi: {file_path}")

    with open(file_path, "r", encoding="utf-8", errors="ignore") as f:
        raw_text = f.read()

    cleaned_text, html_fix_count = strip_html_artifacts(raw_text)

    if html_fix_count > 0:
        log(
            f"\nUYARI: Dosyada {html_fix_count} adet olasi HTML "
            "ankor etiketi (<a href=...>...</a>) tespit edildi ve "
            "temizlendi."
        )

    raw_text = cleaned_text

    records, parse_errors = extract_json_objects(raw_text)

    log(
        "Brace-counting ile bulunan toplam JSON obje sayisi:",
        len(records) + len(parse_errors)
    )
    log("Basariyla parse edilen kayit sayisi:", len(records))

    if parse_errors:

        log(
            f"\nUYARI: {len(parse_errors)} kayit parse edilemedi ve "
            "ATLANDI. Detaylar:"
        )

        for obj_index, error_message, preview in parse_errors:
            log(f"\n  [Obje #{obj_index}] Hata: {error_message}")
            log(f"  Icerik onizleme: {preview}...")

        with open(PAYLOAD_PARSE_ERRORS_PATH, "w", encoding="utf-8") as f:
            f.write(f"Toplam {len(parse_errors)} hatali obje bulundu.\n\n")
            for obj_index, error_message, preview in parse_errors:
                f.write(f"--- Obje #{obj_index} | Hata: {error_message} ---\n")
                f.write(preview + "...\n\n")

        log(
            "\nHatali kayitlarin detayi su dosyaya kaydedildi:",
            PAYLOAD_PARSE_ERRORS_PATH
        )

        log(
            f"\nDevam ediliyor: {len(records)} saglam kayitla egitim "
            "yapilacak (hatali kayitlar egitim disinda birakildi)."
        )

    if not records:
        raise ValueError(
            "Hicbir kayit basariyla parse edilemedi. Dosya formatini "
            "kontrol edin."
        )

    log("\nToplam kullanilacak kayit sayisi (parse sonrasi):", len(records))

    df = pd.DataFrame(records)

    if "id" not in df.columns:
        raise ValueError(
            "Beklenen 'id' kolonu bulunamadi. "
            f"Bulunan kolonlar: {df.columns.tolist()}"
        )

    df["id_prefix"] = df["id"].str.split("-").str[0]
    df["category"] = df["id_prefix"].map(ID_PREFIX_TO_CATEGORY)

    unmapped_mask = df["category"].isna()

    if unmapped_mask.any():
        unmapped_prefixes = df.loc[unmapped_mask, "id_prefix"].unique().tolist()
        log(
            "\nUYARI: Su id onekleri ID_PREFIX_TO_CATEGORY sozlugunde "
            f"bulunamadi ve egitimden cikarilacak: {unmapped_prefixes}"
        )
        df = df[~unmapped_mask].copy()

    if "payload" not in df.columns:
        raise ValueError(
            "Beklenen 'payload' kolonu HICBIR kayitta bulunamadi. "
            f"Bulunan kolonlar: {df.columns.tolist()}"
        )

    missing_payload_mask = df["payload"].isna()

    payload_as_str = df["payload"].astype(str).str.strip()
    empty_after_strip_mask = (
        (payload_as_str == "")
        | (payload_as_str.str.lower() == "none")
        | (payload_as_str.str.lower() == "nan")
    )

    drop_mask = missing_payload_mask | empty_after_strip_mask

    if drop_mask.any():

        dropped_count = int(drop_mask.sum())

        log(
            f"\nUYARI: {dropped_count} kayitta 'payload' alani "
            "eksik/None/bos oldugu icin egitim disinda birakildi. "
            "Dusurulen kayitlarin 'id' degerleri:"
        )

        dropped_ids = df.loc[drop_mask, "id"].tolist()
        log(dropped_ids)

        df.loc[drop_mask, ["id", "category"]].to_csv(
            PAYLOAD_DROPPED_ROWS_PATH, index=False
        )

        log(
            "Dusurulen kayitlarin detayi su dosyaya kaydedildi:",
            PAYLOAD_DROPPED_ROWS_PATH
        )

        df = df[~drop_mask].copy()

    if df.empty:
        raise ValueError(
            "Tum kayitlar filtrelendi (gecerli 'payload' alani "
            "kalmadi). Dosya icerigini kontrol edin."
        )

    df["payload"] = df["payload"].astype(str)

    log("\nEgitimde kullanilacak nihai kayit sayisi:", len(df))

    log("\nKategori dagilimi:")
    log(df["category"].value_counts())

    df["category"].value_counts().rename("sample_count").to_csv(
        PAYLOAD_CATEGORY_SUMMARY_PATH
    )

    df.to_csv(PAYLOAD_TRAINING_DATA_PATH, index=False)

    return df


# ============================================================
# 5. STRATIFIEDKFOLD CROSS-VALIDATION (TF-IDF + EXTRATREES)
# ============================================================

def train_payload_classifier_v2(training_df):
    """
    WEB_APPLICATION_PAYLOADS'daki GERCEK etiketlerle
    (ground-truth) StratifiedKFold ile leakage-safe bir OOF
    degerlendirmesi yapar ve tum veriyle nihai production
    modelini egitir.

    BU SURUMDE YENI: Ozel class_weight kullanir (bkz. dosyanin
    basindaki CUSTOM_CLASS_WEIGHTS_BY_NAME aciklamasi). Command
    Injection'in agirligi dusuruluyor, SSRF/CSRF/XSS'e hafif
    ekstra agirlik veriliyor.
    """

    training_df = training_df.copy()
    training_df["payload"] = training_df["payload"].astype(str)

    valid_mask = (
        training_df["payload"].str.strip().ne("")
        & training_df["payload"].str.strip().str.lower().ne("nan")
        & training_df["payload"].str.strip().str.lower().ne("none")
    )

    if (~valid_mask).any():
        log(
            f"\nUYARI (train_payload_classifier_v2 icinde): "
            f"{int((~valid_mask).sum())} ek gecersiz payload satiri "
            "daha tespit edilip cikarildi."
        )
        training_df = training_df[valid_mask].copy()

    X_text = training_df["payload"].to_numpy()
    y_text = training_df["category"].to_numpy()

    label_encoder = LabelEncoder()
    y_encoded = label_encoder.fit_transform(y_text)

    log("\nSaldiri kategorileri:")
    for class_index, class_name in enumerate(label_encoder.classes_):
        log(class_index, "->", class_name)

    # --------------------------------------------------------
    # OZEL CLASS_WEIGHT: ISIM BAZLI SOZLUGU INT BAZLI SOZLUGE CEVIR
    # --------------------------------------------------------
    # ONEMLI: sklearn'in class_weight parametresi STRING (kategori
    # ismi) DEGIL, LabelEncoder'in urettigi INT indeks bekler.
    # label_encoder.classes_ zaten alfabetik siraya gore int
    # indekslere karsilik gelir (enumerate ile bu eslemeyi
    # kuruyoruz).
    class_weight_dict = {
        class_index: CUSTOM_CLASS_WEIGHTS_BY_NAME.get(class_name, 1.0)
        for class_index, class_name in enumerate(label_encoder.classes_)
    }

    log(
        "\n>>> Kullanilacak class_weight (isim -> int donusturulmus):",
        class_weight_dict
    )

    for class_index, class_name in enumerate(label_encoder.classes_):
        log(
            f"    {class_index} ({class_name}) -> "
            f"agirlik={class_weight_dict[class_index]}"
        )

    class_counts_final = pd.Series(y_encoded).value_counts()
    smallest_class_count = int(class_counts_final.min())
    smallest_class_encoded = int(class_counts_final.idxmin())
    smallest_class_name = label_encoder.inverse_transform([smallest_class_encoded])[0]

    n_splits = min(N_SPLITS_MAX, smallest_class_count)

    if n_splits < N_SPLITS_MIN:
        raise ValueError(
            f"En az orneli kategori '{smallest_class_name}' icinde "
            f"{smallest_class_count} kayit var. StratifiedKFold icin "
            f"en az {N_SPLITS_MIN} gerekir."
        )

    log("\nEn az orneli kategori:", smallest_class_name, "| boyutu:", smallest_class_count)
    log("Kullanilacak fold sayisi:", n_splits)

    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=RANDOM_STATE)

    n_rows = len(X_text)
    oof_pred_encoded = np.full(shape=n_rows, fill_value=-1, dtype=int)
    oof_confidence = np.full(shape=n_rows, fill_value=np.nan, dtype=float)
    fold_metrics_records = []

    log("\n" + "=" * 80)
    log(f"PAYLOAD MODELI v2 STRATIFIEDKFOLD BASLIYOR ({n_splits} fold, n_estimators={CV_N_ESTIMATORS})")
    log("=" * 80)

    for fold_index, (train_idx, test_idx) in enumerate(skf.split(X_text, y_encoded), start=1):

        log(f"\n[Fold {fold_index}/{n_splits}] basliyor...")

        X_train_fold = X_text[train_idx]
        X_test_fold = X_text[test_idx]
        y_train_fold = y_encoded[train_idx]
        y_test_fold = y_encoded[test_idx]

        fold_vectorizer = TfidfVectorizer(
            analyzer="char_wb",
            ngram_range=TFIDF_NGRAM_RANGE,
            max_features=TFIDF_MAX_FEATURES,
            lowercase=True
        )

        X_train_fold_vec = fold_vectorizer.fit_transform(X_train_fold)
        X_test_fold_vec = fold_vectorizer.transform(X_test_fold)

        fold_model = ExtraTreesClassifier(
            n_estimators=CV_N_ESTIMATORS,
            max_features="sqrt",
            min_samples_leaf=2,
            class_weight=class_weight_dict,
            random_state=RANDOM_STATE,
            n_jobs=N_JOBS
        )

        fold_model.fit(X_train_fold_vec, y_train_fold)

        fold_pred = fold_model.predict(X_test_fold_vec)
        fold_proba = fold_model.predict_proba(X_test_fold_vec)

        oof_pred_encoded[test_idx] = fold_pred
        oof_confidence[test_idx] = fold_proba.max(axis=1)

        fold_accuracy = accuracy_score(y_test_fold, fold_pred)
        fold_balanced_accuracy = balanced_accuracy_score(y_test_fold, fold_pred)
        fold_macro_f1 = f1_score(y_test_fold, fold_pred, average="macro", zero_division=0)

        fold_metrics_records.append({
            "fold": fold_index,
            "train_size": len(train_idx),
            "test_size": len(test_idx),
            "accuracy": fold_accuracy,
            "balanced_accuracy": fold_balanced_accuracy,
            "macro_f1": fold_macro_f1
        })

        log(
            f"Fold {fold_index}/{n_splits} | train={len(train_idx)} | "
            f"test={len(test_idx)} | Acc={fold_accuracy:.4f} | "
            f"BalAcc={fold_balanced_accuracy:.4f} | MacroF1={fold_macro_f1:.4f}"
        )

        del X_train_fold_vec, X_test_fold_vec, fold_vectorizer, fold_model, fold_pred, fold_proba
        gc.collect()

    assert (oof_pred_encoded != -1).all(), "Bazi satirlar hicbir fold icinde test edilmedi."

    fold_metrics_df = pd.DataFrame(fold_metrics_records)
    fold_metrics_df.to_csv(PAYLOAD_FOLD_METRICS_PATH, index=False)

    log("\nFold bazli metrikler:")
    log(fold_metrics_df)

    overall_accuracy = accuracy_score(y_encoded, oof_pred_encoded)
    overall_balanced_accuracy = balanced_accuracy_score(y_encoded, oof_pred_encoded)
    overall_macro_f1 = f1_score(y_encoded, oof_pred_encoded, average="macro", zero_division=0)

    log("\n" + "=" * 80)
    log("PAYLOAD MODELI v2 - OOF GENEL SONUCLAR")
    log("=" * 80)
    log("Accuracy:", round(overall_accuracy, 4))
    log("Balanced Accuracy:", round(overall_balanced_accuracy, 4))
    log("Macro F1:", round(overall_macro_f1, 4))

    report_text = classification_report(
        y_encoded, oof_pred_encoded,
        labels=np.arange(len(label_encoder.classes_)),
        target_names=label_encoder.classes_,
        zero_division=0, digits=4
    )

    log("\nOOF Classification Report:\n")
    log(report_text)

    report_dict = classification_report(
        y_encoded, oof_pred_encoded,
        labels=np.arange(len(label_encoder.classes_)),
        target_names=label_encoder.classes_,
        zero_division=0, output_dict=True
    )

    pd.DataFrame(report_dict).transpose().to_csv(PAYLOAD_REPORT_PATH, index=True)

    cm = confusion_matrix(y_encoded, oof_pred_encoded, labels=np.arange(len(label_encoder.classes_)))

    plt.figure(figsize=(9, 7))
    sns.heatmap(
        cm, annot=True, fmt="d", cmap="Blues",
        xticklabels=label_encoder.classes_, yticklabels=label_encoder.classes_
    )
    plt.title("Payload Classification v2 - OOF Confusion Matrix (class_weight uygulanmis)")
    plt.xlabel("Tahmin Edilen Kategori")
    plt.ylabel("Gercek Kategori")
    plt.xticks(rotation=30, ha="right")
    plt.tight_layout()
    plt.savefig(PAYLOAD_CONFUSION_MATRIX_PATH, dpi=150, bbox_inches="tight")
    plt.close()

    log(f"\nNihai production payload modeli (v2) egitiliyor (n_estimators={FINAL_N_ESTIMATORS})...")

    final_vectorizer = TfidfVectorizer(
        analyzer="char_wb",
        ngram_range=TFIDF_NGRAM_RANGE,
        max_features=TFIDF_MAX_FEATURES,
        lowercase=True
    )

    X_all_vec = final_vectorizer.fit_transform(X_text)

    final_model = ExtraTreesClassifier(
        n_estimators=FINAL_N_ESTIMATORS,
        max_features="sqrt",
        min_samples_leaf=2,
        class_weight=class_weight_dict,
        random_state=RANDOM_STATE,
        n_jobs=N_JOBS
    )

    final_model.fit(X_all_vec, y_encoded)

    log("Nihai payload modeli (v2) egitimi tamamlandi.")

    bundle = {
        "vectorizer": final_vectorizer,
        "model": final_model,
        "label_encoder": label_encoder,

        "task": "web_payload_multiclass_classification_v2",
        "labeling_method": "ground_truth_from_labeled_dataset",

        "categories": list(label_encoder.classes_),

        "fallback_final_label": FALLBACK_FINAL_LABEL,
        "low_confidence_threshold": LOW_CONFIDENCE_THRESHOLD,

        "layer1_closed_world_classes": ["SQL Injection", "XSS"],

        "class_weight_used": class_weight_dict,
        "custom_class_weights_by_name": CUSTOM_CLASS_WEIGHTS_BY_NAME,

        "cv_strategy": "StratifiedKFold",
        "cv_n_splits": n_splits,

        "metrics_oof": {
            "accuracy": float(overall_accuracy),
            "balanced_accuracy": float(overall_balanced_accuracy),
            "macro_f1": float(overall_macro_f1)
        },

        "metrics_per_fold": fold_metrics_df.to_dict(orient="records")
    }

    joblib.dump(bundle, PAYLOAD_MODEL_PATH)
    log("\nPayload model paketi (v2) kaydedildi:", PAYLOAD_MODEL_PATH)

    return bundle


# ============================================================
# 6. PRODUCTION: MODEL YUKLEME VE TAHMIN FONKSIYONLARI
# ============================================================

def load_payload_bundle_v2(model_path=None):
    """Kaydedilmis v2 payload model paketini diskten yukler."""

    if model_path is None:
        model_path = PAYLOAD_MODEL_PATH

    return joblib.load(model_path)


def classify_web_payload_v2(payload_text, bundle):
    """
    GENEL AMACLI (5 sinifli) tahmin: SQL Injection / XSS /
    CSRF / SSRF / Command Injection.
    """

    if payload_text is None:
        payload_text = ""

    payload_text = str(payload_text)

    vectorizer = bundle["vectorizer"]
    model = bundle["model"]
    label_encoder = bundle["label_encoder"]
    low_confidence_threshold = bundle.get("low_confidence_threshold", LOW_CONFIDENCE_THRESHOLD)

    if payload_text.strip() == "":
        return {
            "predicted_category": None,
            "confidence": 0.0,
            "is_low_confidence": True,
            "decision_basis": "Payload bos oldugu icin siniflandirma yapilamadi."
        }

    payload_vector = vectorizer.transform([payload_text])

    predicted_encoded = model.predict(payload_vector)[0]
    probabilities = model.predict_proba(payload_vector)[0]

    predicted_category = label_encoder.inverse_transform([predicted_encoded])[0]
    confidence = float(probabilities.max())

    return {
        "predicted_category": predicted_category,
        "confidence": confidence,
        "is_low_confidence": confidence < low_confidence_threshold,
        "decision_basis": (
            f"ML modeli (v2, ground-truth etiketli veriyle egitilmis) "
            f"'{predicted_category}' kategorisini tespit etti."
        )
    }


def classify_web_payload_for_layer1(payload_text, bundle):
    """
    LAYER 1 UYUMLULUK FONKSIYONU: attack_type_classifier.py'nin
    WebAttackCandidate ciktisi icin kullanilir.
    """

    result = classify_web_payload_v2(payload_text, bundle)

    fallback_label = bundle.get("fallback_final_label", FALLBACK_FINAL_LABEL)
    layer1_classes = bundle.get("layer1_closed_world_classes", ["SQL Injection", "XSS"])

    predicted_category = result["predicted_category"]

    if predicted_category is None:
        return {
            "final_attack_type": fallback_label,
            "confidence": 0.0,
            "source": DECISION_SOURCE_FALLBACK,
            "is_low_confidence": True,
            "decision_basis": "Payload bos; eleme yontemiyle Web Brute Force kabul edildi."
        }

    if predicted_category in layer1_classes and not result["is_low_confidence"]:
        return {
            "final_attack_type": predicted_category,
            "confidence": result["confidence"],
            "source": DECISION_SOURCE_ML,
            "is_low_confidence": False,
            "decision_basis": result["decision_basis"]
        }

    return {
        "final_attack_type": fallback_label,
        "confidence": result["confidence"],
        "source": DECISION_SOURCE_FALLBACK,
        "is_low_confidence": result["is_low_confidence"],
        "decision_basis": (
            f"Model '{predicted_category}' tahmin etti (guven="
            f"{result['confidence']:.2f}), ancak bu Layer 1'in kapali "
            "kumesinde (SQLi/XSS) degil veya guven esigin altinda. "
            "Eleme yontemiyle Web Brute Force kabul edildi."
        )
    }


# ============================================================
# 7. DOGRULAMA TESTI
# ============================================================

def run_sanity_check(bundle):
    """Bilinen orneklerle modelin dogru calistigini hizlica dogrular."""

    test_cases = [
        {"name": "SQLi - tautology", "payload": "' OR '1'='1", "expected": "SQL Injection"},
        {"name": "SQLi - union", "payload": "' UNION SELECT username, password FROM users--", "expected": "SQL Injection"},
        {"name": "XSS - script tag", "payload": "<script>alert('XSS')</script>", "expected": "XSS"},
        {"name": "XSS - onerror", "payload": "<img src=x onerror=alert(1)>", "expected": "XSS"},
        {"name": "CSRF - auto-submit form", "payload": "<form action='https://target.com/x' method='POST'><input name='a' value='b'/></form>", "expected": "CSRF"},
        {"name": "SSRF - internal IP", "payload": "http://127.0.0.1/", "expected": "SSRF"},
        {"name": "CmdInj - semicolon", "payload": "; ls -la", "expected": "Command Injection"},
    ]

    log("\n" + "=" * 80)
    log("PAYLOAD MODELI v2 - DOGRULAMA TESTI")
    log("=" * 80)

    records = []
    correct_count = 0

    for case in test_cases:

        result = classify_web_payload_v2(case["payload"], bundle)
        is_correct = result["predicted_category"] == case["expected"]
        correct_count += int(is_correct)

        status = "OK" if is_correct else "X"

        log(f"\n[{status}] {case['name']}")
        log("  payload  :", case["payload"])
        log("  beklenen :", case["expected"])
        log("  tahmin   :", result["predicted_category"])
        log("  guven    :", round(result["confidence"], 4))

        records.append({
            "test_name": case["name"],
            "payload": case["payload"],
            "expected": case["expected"],
            "predicted": result["predicted_category"],
            "confidence": result["confidence"],
            "is_correct": is_correct
        })

    results_df = pd.DataFrame(records)
    results_df.to_csv(PAYLOAD_SANITY_CHECK_PATH, index=False)

    log("\nDogrulama sonuclari kaydedildi:", PAYLOAD_SANITY_CHECK_PATH)
    log(f"\nDogrulama seti dogrulugu: {correct_count / len(test_cases) * 100:.2f}% ({correct_count}/{len(test_cases)})")

    return results_df


# ============================================================
# 8. EGITIM SCRIPTINI CALISTIR
# ============================================================

if __name__ == "__main__":

    training_dataframe = load_payload_json()
    trained_bundle = train_payload_classifier_v2(training_dataframe)

    log("\n" + "=" * 80)
    log("PAYLOAD SINIFLANDIRMA PIPELINE v2 TAMAMLANDI (LAYER 2)")
    log("=" * 80)

    log("Model:", PAYLOAD_MODEL_PATH)
    log("OOF report:", PAYLOAD_REPORT_PATH)
    log("Confusion matrix:", PAYLOAD_CONFUSION_MATRIX_PATH)
    log("Kategori ozeti:", PAYLOAD_CATEGORY_SUMMARY_PATH)

    log(
        "\nNOT: classify_web_payload_for_layer1() fonksiyonu, mevcut "
        "SOC pipeline'i (attack_type_classifier.py - WebAttackCandidate) "
        "ile geriye donuk uyumludur. Genel amacli 5 sinifli tahmin icin "
        "classify_web_payload_v2() kullanin."
    )

    run_sanity_check(trained_bundle)



Payload JSON/JSONL dosyasi okunuyor: data/WEB_APPLICATION_PAYLOADS.jsonl

UYARI: Dosyada 5 adet olasi HTML ankor etiketi (<a href=...>...</a>) tespit edildi ve temizlendi.
Brace-counting ile bulunan toplam JSON obje sayisi: 500
Basariyla parse edilen kayit sayisi: 480

UYARI: 20 kayit parse edilemedi ve ATLANDI. Detaylar:

  [Obje #19] Hata: Expecting ',' delimiter: line 9 column 1 (char 265)
  Icerik onizleme: {     "id": "sqli-020",     "description": "Using OR with false condition and comment",     "payload": "' OR 1=1#",     "context": "Login form",     "type": "tautology",     "severity": "high",     "e...

  [Obje #39] Hata: Expecting ',' delimiter: line 9 column 1 (char 394)
  Icerik onizleme: {     "id": "sqli-040",     "description": "Using stacked queries to insert data",     "payload": "'; INSERT INTO users (username, password) VALUES ('attacker', 'pass')--",     "context": "User input"...

  [Obje #69] Hata: Expecting ',' delimiter: line 9 column 1 (char 388)
  Icerik oniz